<a href="https://colab.research.google.com/github/atman500/AI-Academic-Evaluator/blob/main/Sentimental%20anlysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from google.colab import files

In [15]:
files_1=files.upload()

Saving reuters_headlines.csv to reuters_headlines (1).csv
Saving guardian_headlines.csv to guardian_headlines.csv
Saving cnbc_headlines.csv to cnbc_headlines (1).csv


In [21]:
print("Reuters Headlines Descriptions (first 5):")
if 'description' in reuters_df.columns:
    display(reuters_df['description'].head())
else:
    print("No 'description' column found in reuters_df.")

print("\nCNBC Headlines Descriptions (first 5):")
if 'description' in cnbc_df.columns:
    display(cnbc_df['description'].head())
else:
    print("No 'description' column found in cnbc_df.")

print("\nGuardian Headlines Descriptions (first 5):")
if 'description' in guardian_df.columns:
    display(guardian_df['description'].head())
else:
    print("No 'description' column found in guardian_df.")

Reuters Headlines Descriptions (first 5):


,description
0,TikTok has been in discussions with the UK gov...
1,Walt Disney has become the latest company to ...
2,Jul 18 2020
3,Twitter Inc said on Saturday that hackers were...
4,A battle in the U.S. Congress over a new coron...



CNBC Headlines Descriptions (first 5):


,description
0,"""Mad Money"" host Jim Cramer recommended buying..."
1,"""Mad Money"" host Jim Cramer rings the lightnin..."
2,NaN
3,"7:25 PM ET Fri, 17 July 2020"
4,"Keith Bliss, IQ Capital CEO, joins ""Closing Be..."



Guardian Headlines Descriptions (first 5):


,description
0,NaN
1,NaN
2,NaN
3,NaN
4,NaN


In [24]:
import io
import re

# Read reuters_headlines.csv
reuters_df = pd.read_csv(io.StringIO(files_1['reuters_headlines (1).csv'].decode('utf-8')), sep=';')
reuters_df = reuters_df.rename(columns={'Headlines': 'headline', 'Time': 'time', 'Description': 'description'})
# Drop unnamed columns that might result from extra semicolons
reuters_df = reuters_df.loc[:, ~reuters_df.columns.str.contains('^Unnamed')]

# Read cnbc_headlines.csv
cnbc_df = pd.read_csv(io.StringIO(files_1['cnbc_headlines (1).csv'].decode('utf-8')), sep=';')
cnbc_df = cnbc_df.rename(columns={'Headlines': 'headline', 'Time': 'time', 'Description': 'description'})
# Drop unnamed columns
cnbc_df = cnbc_df.loc[:, ~cnbc_df.columns.str.contains('^Unnamed')]

# Read guardian_headlines.csv, attempting 'latin1' encoding for special characters
guardian_df = pd.read_csv(io.StringIO(files_1['guardian_headlines.csv'].decode('latin1')), sep=';')
# Guardian has 'Time' then 'Headlines', so swap them and rename
guardian_df = guardian_df.rename(columns={'Headlines': 'headline', 'Time': 'time'})
# Drop unnamed columns
guardian_df = guardian_df.loc[:, ~guardian_df.columns.str.contains('^Unnamed')]

# --- Cleaning 'time' column for guardian_df ---
def clean_guardian_time(time_str):
    if pd.isna(time_str):
        return time_str
    # Attempt to replace common garbled month patterns
    time_str = time_str.replace('íæáíæ', 'Jul') # Likely July
    time_str = time_str.replace('ÏíÓãÈÑ', 'Sep') # Likely September
    return time_str

guardian_df['time'] = guardian_df['time'].apply(clean_guardian_time)

# Ensure all dataframes have the same columns before concatenating
# We'll keep 'headline' and 'time' for all, and 'description' if present
common_cols = ['headline', 'time']

# Add 'description' column to guardian_df if it doesn't exist, fill with NaN
if 'description' not in guardian_df.columns:
    guardian_df['description'] = np.nan

# Select common columns for all dataframes
reuters_df = reuters_df[common_cols + ['description']]
cnbc_df = cnbc_df[common_cols + ['description']]
guardian_df = guardian_df[common_cols + ['description']]

# Convert time columns to datetime after cleaning
reuters_df['time'] = pd.to_datetime(reuters_df['time'], errors='coerce')

# Clean CNBC time column by removing timezone info before conversion
cnbc_df['time'] = cnbc_df['time'].astype(str).str.replace(r'\s*PM ET|\s*AM ET|\s*ET', '', regex=True)
cnbc_df['time'] = pd.to_datetime(cnbc_df['time'], errors='coerce')

guardian_df['time'] = pd.to_datetime(guardian_df['time'], format='%d-%b-%y', errors='coerce')

# Merge all three dataframes
df = pd.concat([reuters_df, cnbc_df, guardian_df], ignore_index=True)

display(df.head())

,headline,time,description
0,TikTok considers London and other locations fo...,2020-07-18,TikTok has been in discussions with the UK gov...
1,Disney cuts ad spending on Facebook amid growi...,2020-07-18,Walt Disney has become the latest company to ...
2,Trail of missing Wirecard executive leads to B...,NaT,Jul 18 2020
3,Twitter says attackers downloaded data from up...,2020-07-18,Twitter Inc said on Saturday that hackers were...
4,U.S. Republicans seek liability protections as...,2020-07-17,A battle in the U.S. Congress over a new coron...


In [22]:
print(df)

                                                headline  \
0      TikTok considers London and other locations fo...   
1      Disney cuts ad spending on Facebook amid growi...   
2      Trail of missing Wirecard executive leads to B...   
3      Twitter says attackers downloaded data from up...   
4      U.S. Republicans seek liability protections as...   
...                                                  ...   
57384  How investing in solar energy can create a bri...   
57385     Poundland suppliers hit by insurance downgrade   
57386  Cryptocurrencies: City watchdog to investigate...   
57387  Unilever sells household name spreads to KKR f...   
57388  The Guardian view on Ryanairâs model: a unio...   

                       time                                        description  
0               Jul 18 2020  TikTok has been in discussions with the UK gov...  
1               Jul 18 2020  Walt Disney  has become the latest company to ...  
2       Der Spiegel reports         

In [25]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57389 entries, 0 to 57388
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   headline     53549 non-null  object        
 1   time         26918 non-null  datetime64[ns]
 2   description  34995 non-null  object        
dtypes: datetime64[ns](1), object(2)
memory usage: 1.3+ MB
None


In [26]:
print(df.describe())

                                time
count                          26918
mean   2019-05-28 21:53:52.470465792
min              2017-09-17 00:00:00
25%              2018-10-05 00:00:00
50%              2019-06-12 00:00:00
75%              2020-01-20 00:00:00
max              2020-07-18 00:00:00
